In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import hashlib

# ============================================================
# 0) CONEXIÓN POSTGRES
# ============================================================
usuario = "andrus"
contraseña = "123"
host = "localhost"
puerto = "5432"
base = "andres"

engine = create_engine(f"postgresql://{usuario}:{contraseña}@{host}:{puerto}/{base}")

def subir(df: pd.DataFrame, tabla: str):
    df.to_sql(tabla, engine, if_exists="replace", index=False, method="multi", chunksize=20000)
    print(f"✅ {tabla}: {len(df):,} filas")

# ============================================================
# 1) RUTAS
# ============================================================
ruta_fuentes = Path(r"D:\Documentos\Repositorio\Fuentes_originales")

# ============================================================
# 2) HELPERS
# ============================================================
def cargar_csv(ruta: Path) -> pd.DataFrame:
    for enc in ["utf-8", "utf-8-sig", "latin-1", "cp1252"]:
        try:
            return pd.read_csv(ruta, encoding=enc, sep=";")
        except Exception:
            pass
    raise ValueError(f"No se pudo leer {ruta.name}")

def normalizar_columnas(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    return df

def safe_str(df, col):
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

def norm_code(s):
    return (
        s.astype(str)
         .str.strip()
         .replace({"nan": np.nan, "None": np.nan})
         .str.replace(r"\.0$", "", regex=True)
    )

def to_num(s):
    x = s.astype(str).str.strip()
    x = x.str.replace("\u00a0", " ", regex=False)
    x = x.str.replace(" ", "", regex=False)
    x = x.str.replace(".", "", regex=False)   # miles
    x = x.str.replace(",", ".", regex=False)  # decimal
    x = x.str.replace(r"[^\d\.\-]", "", regex=True)
    return pd.to_numeric(x, errors="coerce")

def make_tx_id(df: pd.DataFrame, cols: list) -> pd.Series:
    use = [c for c in cols if c in df.columns]
    tmp = df[use].copy()
    for c in use:
        tmp[c] = tmp[c].astype(str).fillna("")
    base = tmp.apply(lambda r: "|".join(r.values.tolist()), axis=1)
    return base.apply(lambda x: hashlib.md5(x.encode("utf-8")).hexdigest())

def print_profile(df: pd.DataFrame, name: str, date_col: str = None, num_cols: list = None, cat_cols: list = None):
    print(f"\n==================== PERFIL {name} ====================")
    print(f"Filas: {len(df):,} | Columnas: {df.shape[1]:,}")

    if date_col and date_col in df.columns:
        dmin = pd.to_datetime(df[date_col], errors="coerce").min()
        dmax = pd.to_datetime(df[date_col], errors="coerce").max()
        print(f"Rango {date_col}: {dmin} -> {dmax} | Nulos: {df[date_col].isna().sum():,}")

    if num_cols:
        for c in num_cols:
            if c in df.columns:
                s = pd.to_numeric(df[c], errors="coerce")
                print(f"- {c}: nulos={s.isna().sum():,} | p50={s.quantile(.50)} | p95={s.quantile(.95)} | max={s.max()}")

    if cat_cols:
        for c in cat_cols:
            if c in df.columns:
                print(f"- Top {c}:")
                print(df[c].astype(str).value_counts(dropna=False).head(5).to_string())

def require_columns(df: pd.DataFrame, cols: list, name: str):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"❌ {name}: faltan columnas requeridas: {missing}")

# -------------------------
# SUBIDAS
# -------------------------
def subir_replace(df: pd.DataFrame, tabla: str):
    df.to_sql(tabla, engine, if_exists="replace", index=False, method="multi", chunksize=5000)
    print(f"✅ {tabla} (REPLACE): {len(df):,} filas")

def subir_append(df: pd.DataFrame, tabla: str, chunksize=20000):
    if df is None or df.empty:
        print(f"⚡ {tabla} (APPEND): 0 filas (sin nuevos)")
        return
    df.to_sql(tabla, engine, if_exists="append", index=False, method="multi", chunksize=chunksize)
    print(f"✅ {tabla} (APPEND): {len(df):,} filas insertadas")

# -------------------------
# INCREMENTAL FACT: fecha + tx_id (sin leer toda la tabla)
# -------------------------
def fact_incremental_por_fecha_y_txid(fact: pd.DataFrame, buffer_dias: int = 2) -> pd.DataFrame:
    """
    Devuelve SOLO las transacciones realmente nuevas (fact_nuevo),
    usando:
      - última fecha cargada en fact_transaccion
      - buffer de seguridad en días
      - tx_id existentes desde min_fecha candidata
    """
    # 1) última fecha
    try:
        x = pd.read_sql("SELECT MAX(txfecha) AS max_fecha FROM fact_transaccion", engine)
        ultima_fecha = x["max_fecha"].iloc[0]
    except Exception:
        ultima_fecha = None

    # 2) candidatos por fecha (con buffer)
    if ultima_fecha is not None and pd.notna(ultima_fecha):
        corte = pd.to_datetime(ultima_fecha) - pd.Timedelta(days=buffer_dias)
        candidatos = fact[fact["txfecha"] > corte].copy()
        print(f"📅 Última txfecha en BD: {ultima_fecha} | Buffer: {buffer_dias} días | Candidatos: {len(candidatos):,}")
    else:
        candidatos = fact.copy()
        print(f"📅 Primera carga (o tabla vacía). Candidatos: {len(candidatos):,}")

    if candidatos.empty:
        return candidatos

    # 3) tx_id existentes SOLO desde min_fecha candidata (evita traer toda la tabla)
    min_fecha = pd.to_datetime(candidatos["txfecha"].min())
    q = f"""
        SELECT tx_id
        FROM fact_transaccion
        WHERE txfecha >= '{min_fecha.strftime('%Y-%m-%d %H:%M:%S')}'
    """
    try:
        tx_exist = set(pd.read_sql(q, engine)["tx_id"].astype(str))
    except Exception:
        tx_exist = set()

    fact_nuevo = candidatos[~candidatos["tx_id"].astype(str).isin(tx_exist)].copy()
    print(f"🧩 fact_nuevo (reales nuevos por tx_id): {len(fact_nuevo):,}")
    return fact_nuevo

# -------------------------
# INCREMENTAL AUD: filtra contra existentes por llave (sin romper por unique)
# -------------------------
def filtrar_existentes_por_llave(df: pd.DataFrame, tabla: str, cols_llave: list) -> pd.DataFrame:
    """
    Retorna df sin las filas cuya llave ya exista en Postgres.
    Lee únicamente las llaves existentes (solo esas columnas).
    """
    if df is None or df.empty:
        return df

    # Asegurar strings en llaves para match consistente
    x = df.copy()
    for c in cols_llave:
        x[c] = x[c].astype(str)

    try:
        sql_cols = ", ".join(cols_llave)
        existentes = pd.read_sql(f"SELECT {sql_cols} FROM {tabla}", engine)
        for c in cols_llave:
            existentes[c] = existentes[c].astype(str)

        # merge anti-join
        m = x.merge(existentes.drop_duplicates(), on=cols_llave, how="left", indicator=True)
        out = m[m["_merge"] == "left_only"].drop(columns=["_merge"])
        return out
    except Exception:
        # Si la tabla no existe aún, no filtramos
        return x

# ============================================================
# 3) EXTRAER
# ============================================================
trans = normalizar_columnas(cargar_csv(ruta_fuentes / "TRANSACCIONES.csv"))
cod   = normalizar_columnas(cargar_csv(ruta_fuentes / "CODIGOS_TRANSACCION.csv"))
prod  = normalizar_columnas(cargar_csv(ruta_fuentes / "PRODUCTOS.csv"))
ofis  = normalizar_columnas(cargar_csv(ruta_fuentes / "OFICINAS.csv"))
cli   = normalizar_columnas(cargar_csv(ruta_fuentes / "DATOS_CLIENTES.csv"))
tipid = normalizar_columnas(cargar_csv(ruta_fuentes / "TIPO_IDENTIFICACION.csv"))

print("✅ Fuentes cargadas")


✅ Fuentes cargadas


In [ ]:
# 4) LIMPIEZA MÍNIMA + TIPOS
# ============================================================
require_columns(trans, ["txfecha", "ctxcodigo", "cccuentanum"], "TRANSACCIONES")
require_columns(cod,   ["ctxcodigo"], "CODIGOS_TRANSACCION")
require_columns(prod,  ["cccuentanum"], "PRODUCTOS")

# STRINGS - TRANS
for c in ["cccuentanum","ctxcodigoproducto","usuario","cargo","ofcodigo","txoficinacuenta","ctxcodigo","horas"]:
    safe_str(trans, c)

# STRINGS - COD
for c in ["ctxcodigo","descripcion_ctxcodigo","requiere_presencia_cliente"]:
    safe_str(cod, c)

# STRINGS - PROD
for c in ["cccuentanum","ctxcodigoproducto","usuario_apertura","txoficinacuenta","clidcliente","clidtipo"]:
    safe_str(prod, c)

# NORMALIZAR CÓDIGOS
for df, col in [
    (trans,"cccuentanum"), (prod,"cccuentanum"),
    (trans,"ctxcodigoproducto"), (prod,"ctxcodigoproducto"),
    (trans,"ctxcodigo"), (cod,"ctxcodigo"),
    (trans,"ofcodigo"), (trans,"txoficinacuenta")
]:
    if col in df.columns:
        df[col] = norm_code(df[col])

# FECHAS
trans["txfecha"] = pd.to_datetime(trans["txfecha"], errors="coerce", dayfirst=True)
if "txfecha_apertura" in prod.columns:
    prod["txfecha_apertura"] = pd.to_datetime(prod["txfecha_apertura"], errors="coerce", dayfirst=True)

# NUMÉRICOS
if "txefectivo" in trans.columns: trans["txefectivo"] = to_num(trans["txefectivo"])
if "txtotal" in trans.columns:    trans["txtotal"] = to_num(trans["txtotal"])
if "monto" in prod.columns:       prod["monto"] = to_num(prod["monto"])
if "plazo" in prod.columns:       prod["plazo"] = pd.to_numeric(prod["plazo"], errors="coerce")

# PRESENCIA (SI/NO)
if "requiere_presencia_cliente" in cod.columns:
    cod["requiere_presencia_cliente"] = cod["requiere_presencia_cliente"].astype(str).str.strip().str.upper()

# ============================================================
# 5) PERFILAMIENTO
# ============================================================
print_profile(trans, "TRANSACCIONES", date_col="txfecha", num_cols=["txtotal","txefectivo"], cat_cols=["ofcodigo","cargo","ctxcodigoproducto"])
print_profile(prod,  "PRODUCTOS", date_col="txfecha_apertura", num_cols=["monto","plazo"], cat_cols=["ctxcodigoproducto"])
print_profile(cod,   "CODIGOS_TRANSACCION", cat_cols=["requiere_presencia_cliente"])

# ============================================================
# 6) HOMOLOGAR ctxcodigoproducto EN PRODUCTOS (fuente confiable = TRANSACCIONES)
# ============================================================
if "ctxcodigoproducto" in trans.columns:
    trans_map = (
        trans.dropna(subset=["cccuentanum","ctxcodigoproducto"])
             .groupby("cccuentanum")["ctxcodigoproducto"]
             .agg(lambda s: s.value_counts().index[0])
             .reset_index()
             .rename(columns={"ctxcodigoproducto":"columna_homologada"})
    )
else:
    trans_map = pd.DataFrame(columns=["cccuentanum","columna_homologada"])

prod = prod.merge(trans_map, on="cccuentanum", how="left")

map_tipo = {"76":"CDT Tasa Fija", "85":"Cuentamiga Digital"}
prod["tipo_producto"] = prod["columna_homologada"].map(map_tipo)

print("\n✅ tipo_producto en PRODUCTOS:")
print(prod["tipo_producto"].value_counts(dropna=False))

# ============================================================
# 7) ANOMALÍAS DE APERTURA (RECOMENDADO: REPLACE, suele ser más pequeño)
# ============================================================
if "txfecha_apertura" in prod.columns:
    prod["mes_apertura"] = prod["txfecha_apertura"].dt.to_period("M").astype(str)
else:
    prod["mes_apertura"] = np.nan

cols_ap = [
    "mes_apertura","txfecha_apertura","tipo_producto","cccuentanum",
    "clidcliente","clidtipo","usuario_apertura","txoficinacuenta",
    "plazo","monto",
    "regla_id","regla_nombre","severidad","detalle"
]

anom_ap = []

def add_ap(df, regla_id, regla_nombre, severidad, detalle):
    if df is None or len(df) == 0:
        return
    x = df.copy()
    x["regla_id"] = regla_id
    x["regla_nombre"] = regla_nombre
    x["severidad"] = severidad
    x["detalle"] = detalle
    for c in cols_ap:
        if c not in x.columns:
            x[c] = np.nan
    anom_ap.append(x[cols_ap])

# AP01: CDT monto mínimo
if "monto" in prod.columns:
    add_ap(
        prod[(prod["tipo_producto"] == "CDT Tasa Fija") & (prod["monto"].notna()) & (prod["monto"] < 500000)],
        "AP01",
        "Apertura CDT por debajo del monto mínimo ($500.000)",
        "ALTA",
        "Incumplimiento de condiciones de producto: monto mínimo."
    )

# AP02: CDT plazo 80–540 días
if "plazo" in prod.columns:
    add_ap(
        prod[(prod["tipo_producto"] == "CDT Tasa Fija") & (prod["plazo"].notna()) & ((prod["plazo"] < 60) | (prod["plazo"] > 540))],
        "AP02",
        "Apertura CDT con plazo fuera de rango (80–540 días)",
        "ALTA",
        "Incumplimiento de condiciones de producto: plazo permitido."
    )

aud_anomalias_apertura = pd.concat(anom_ap, ignore_index=True) if len(anom_ap) else pd.DataFrame(columns=cols_ap)
print("\n✅ aud_anomalias_apertura:", len(aud_anomalias_apertura))

# ============================================================
# 8) FACT TRANSACCION ENRIQUECIDO (para construir fact)
# ============================================================
cod_m = cod.drop_duplicates(subset=["ctxcodigo"]).copy()

fact = trans.merge(
    cod_m[[c for c in ["ctxcodigo","descripcion_ctxcodigo","requiere_presencia_cliente"] if c in cod_m.columns]],
    on="ctxcodigo",
    how="left"
)

fact["mes_tx"] = fact["txfecha"].dt.to_period("M").astype(str)

tx_id_cols = ["txfecha","horas","usuario","ofcodigo","cccuentanum","ctxcodigo","txtotal","txefectivo"]
fact["tx_id"] = make_tx_id(fact, tx_id_cols)

# ============================================================
# 8B) CONTROL CRUCE
# ============================================================
for col in ["descripcion_ctxcodigo", "requiere_presencia_cliente"]:
    if col not in fact.columns:
        fact[col] = np.nan

n_total = len(fact)
n_sin_desc = fact["descripcion_ctxcodigo"].isna().sum()
n_sin_pres = fact["requiere_presencia_cliente"].isna().sum()

print("\n📌 CONTROL CRUCE CODIGOS_TRANSACCION (ctxcodigo):")
print(f"Total transacciones: {n_total:,}")
print(f"Sin descripcion_ctxcodigo: {n_sin_desc:,} ({(n_sin_desc/n_total if n_total else 0):.2%})")
print(f"Sin requiere_presencia_cliente: {n_sin_pres:,} ({(n_sin_pres/n_total if n_total else 0):.2%})")

# ============================================================
# 8C) INCREMENTAL: construir fact_nuevo (solo lo realmente nuevo)
# ============================================================
fact_nuevo = fact_incremental_por_fecha_y_txid(fact, buffer_dias=2)

# ============================================================
# 9) ANOMALÍAS TRANSACCIONALES SOLO SOBRE fact_nuevo (NO histórico)
# ============================================================
cols_tx = [
    "tx_id","mes_tx","txfecha","horas","cccuentanum","usuario","cargo",
    "ofcodigo","txoficinacuenta",
    "ctxcodigo","descripcion_ctxcodigo","requiere_presencia_cliente",
    "txefectivo","txtotal",
    "regla_id","regla_nombre","severidad","detalle"
]

anom_tx = []

def add_tx(df, regla_id, regla_nombre, severidad, detalle_series):
    if df is None or len(df) == 0:
        return
    x = df.copy()
    x["regla_id"] = regla_id
    x["regla_nombre"] = regla_nombre
    x["severidad"] = severidad
    x["detalle"] = detalle_series
    for c in cols_tx:
        if c not in x.columns:
            x[c] = np.nan
    anom_tx.append(x[cols_tx])

base_tx = fact_nuevo.copy()

if base_tx.empty:
    aud_anomalias_tx_detalle = pd.DataFrame(columns=cols_tx)
    aud_anomalias_tx = pd.DataFrame(columns=[
        "tx_id","mes_tx","txfecha","usuario","ofcodigo","cccuentanum",
        "reglas_ids","reglas_nombres","n_reglas","severidad","detalle_ejemplo"
    ])
    print("\n⚡ No hay transacciones nuevas: no se generan anomalías nuevas TX.")
else:
    cargo_u = base_tx.get("cargo", "").astype(str).str.upper().fillna("")
    pres_u  = base_tx.get("requiere_presencia_cliente", "NO").astype(str).str.upper().fillna("NO")
    en_oficina = base_tx["ofcodigo"].notna() & (base_tx["ofcodigo"].astype(str).str.strip() != "")

    # TX01: efectivo por no cajero
    if "txefectivo" in base_tx.columns:
        cond1 = (base_tx["txefectivo"].fillna(0) > 0) & (~cargo_u.str.contains(r"CAJ|CAJERO|CAJA", na=False))
        det1 = (
            "Efectivo=" + base_tx["txefectivo"].fillna(0).astype(str)
            + " | Cargo=" + base_tx.get("cargo","").astype(str)
            + " | Usuario=" + base_tx.get("usuario","").astype(str)
            + " | Oficina TX=" + base_tx.get("ofcodigo","").astype(str)
            + " | CTX=" + base_tx.get("ctxcodigo","").astype(str)
        )
        add_tx(
            base_tx[cond1],
            "TX01",
            "Efectivo ejecutado por rol NO Cajero (potencial incumplimiento de segregación)",
            "ALTA",
            det1[cond1]
        )

    # TX02: requiere presencia + oficina distinta
    cond2 = (pres_u == "SI") & base_tx["ofcodigo"].notna() & base_tx["txoficinacuenta"].notna() & (base_tx["ofcodigo"] != base_tx["txoficinacuenta"])
    det2 = (
        "Presencia=SI"
        + " | Oficina TX=" + base_tx["ofcodigo"].astype(str)
        + " | Oficina Cuenta=" + base_tx["txoficinacuenta"].astype(str)
        + " | CTX=" + base_tx["ctxcodigo"].astype(str)
        + " | Desc=" + base_tx["descripcion_ctxcodigo"].astype(str)
    )
    add_tx(
        base_tx[cond2],
        "TX02",
        "Transacción que REQUIERE presencia ejecutada en oficina distinta a la oficina de la cuenta",
        "ALTA",
        det2[cond2]
    )

    # TX03: NO requiere presencia pero se hace en oficina
    cond3 = (pres_u == "NO") & en_oficina
    det3 = (
        "Presencia=NO"
        + " | Ejecutada en oficina (proxy=ofcodigo informado)"
        + " | Oficina TX=" + base_tx["ofcodigo"].astype(str)
        + " | CTX=" + base_tx["ctxcodigo"].astype(str)
        + " | Desc=" + base_tx["descripcion_ctxcodigo"].astype(str)
    )
    add_tx(
        base_tx[cond3],
        "TX03",
        "Transacción que NO requiere presencia realizada en oficina (potencial desvío de canal / eficiencia)",
        "BAJA",
        det3[cond3]
    )
# TX04: oficina distinta + señal (efectivo o presencia)
    cond4 = base_tx["ofcodigo"].notna() & base_tx["txoficinacuenta"].notna() & (base_tx["ofcodigo"] != base_tx["txoficinacuenta"]) & (
        (base_tx.get("txefectivo", 0).fillna(0) > 0) | (pres_u == "SI")
    )
    det4 = (
        "Oficina TX=" + base_tx["ofcodigo"].astype(str)
        + " | Oficina Cuenta=" + base_tx["txoficinacuenta"].astype(str)
        + " | Señal=" + np.where(base_tx.get("txefectivo",0).fillna(0) > 0, "EFECTIVO", "PRESENCIA")
        + " | Usuario=" + base_tx.get("usuario","").astype(str)
    )
    add_tx(
        base_tx[cond4],
        "TX04",
        "Transacción en oficina distinta a la oficina de la cuenta con señal de riesgo (efectivo/presencia)",
        "MEDIA",
        det4[cond4]
    )

    # ============================================================
    # TX05 (ALTA): ctxcodigo NO existe en maestro CODIGOS_TRANSACCION
    # Se detecta porque no hay descripcion_ctxcodigo tras el merge
    # ============================================================
    cond5 = base_tx["descripcion_ctxcodigo"].isna() | (base_tx["descripcion_ctxcodigo"].astype(str).str.strip() == "")
    det5 = (
        "CTX=" + base_tx["ctxcodigo"].astype(str)
        + " | No existe en CODIGOS_TRANSACCION (sin descripción/presencia)"
        + " | Usuario=" + base_tx.get("usuario","").astype(str)
        + " | Oficina TX=" + base_tx.get("ofcodigo","").astype(str)
        + " | Cuenta=" + base_tx.get("cccuentanum","").astype(str)
    )
    add_tx(
        base_tx[cond5],
        "TX05",
        "Transacción con código (ctxcodigo) NO encontrado en maestro de códigos",
        "ALTA",
        det5[cond5]
    )

    # --- aquí ya viene tu concat (déjalo igual)
    aud_anomalias_tx_detalle = pd.concat(anom_tx, ignore_index=True) if len(anom_tx) else pd.DataFrame(columns=cols_tx)
    print("\n✅ aud_anomalias_tx_detalle (nuevas):", len(aud_anomalias_tx_detalle))



    # ============================================================
    # 10) CONSOLIDADA: 1 tx_id = 1 fila (solo nuevas)
    # ============================================================
    sev_rank = {"BAJA": 1, "MEDIA": 2, "ALTA": 3}
    txd = aud_anomalias_tx_detalle.copy()
    if not txd.empty:
        txd["sev_n"] = txd["severidad"].map(sev_rank).fillna(0).astype(int)

        aud_anomalias_tx = (
            txd.groupby("tx_id", as_index=False)
               .agg(
                   mes_tx=("mes_tx","max"),
                   txfecha=("txfecha","max"),
                   usuario=("usuario","max"),
                   ofcodigo=("ofcodigo","max"),
                   cccuentanum=("cccuentanum","max"),
                   reglas_ids=("regla_id", lambda x: " | ".join(sorted(set(x)))),
                   reglas_nombres=("regla_nombre", lambda x: " | ".join(sorted(set(x)))),
                   n_reglas=("regla_id","nunique"),
                   sev_max=("sev_n","max"),
                   detalle_ejemplo=("detalle","first")
               )
        )

        inv_sev = {1:"BAJA", 2:"MEDIA", 3:"ALTA"}
        aud_anomalias_tx["severidad"] = aud_anomalias_tx["sev_max"].map(inv_sev).fillna("BAJA")
        aud_anomalias_tx = aud_anomalias_tx.drop(columns=["sev_max"])
    else:
        aud_anomalias_tx = pd.DataFrame(columns=[
            "tx_id","mes_tx","txfecha","usuario","ofcodigo","cccuentanum",
            "reglas_ids","reglas_nombres","n_reglas","severidad","detalle_ejemplo"
        ])

    print("✅ aud_anomalias_tx (consolidada nuevas):", len(aud_anomalias_tx))

# ============================================================
# 11) ALERTA OPERATIVA: USUARIO EN MÚLTIPLES OFICINAS (RECOMENDADO: REPLACE)
# ============================================================
# (Esto se puede optimizar luego por ventana; por ahora, recomputar es más sencillo y confiable)
u = fact.groupby("usuario").agg(
    n_transacciones=("usuario","size"),
    n_oficinas=("ofcodigo","nunique"),
    oficinas_distintas=("ofcodigo", lambda s: ", ".join(sorted(set([x for x in s.dropna().astype(str)]))[:15])),
    fecha_primera=("txfecha","min"),
    fecha_ultima=("txfecha","max"),
    n_meses=("mes_tx","nunique")
).reset_index()

aud_alertas_usuario = u[u["n_oficinas"] >= 2].copy()
aud_alertas_usuario["regla_id"] = "OP01"
aud_alertas_usuario["regla_nombre"] = "Usuario operando en múltiples oficinas (potencial uso compartido de credenciales)"
aud_alertas_usuario["severidad"] = "MEDIA"
aud_alertas_usuario["detalle"] = "Usuarios con transacciones registradas en 2+ oficinas; revisar asignación de roles/perfiles y trazabilidad de accesos."
print("✅ aud_alertas_usuario:", len(aud_alertas_usuario))

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_9036\2341548910.py:30: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S.%f format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  trans["txfecha"] = pd.to_datetime(trans["txfecha"], errors="coerce", dayfirst=True)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_9036\2341548910.py:32: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S.%f format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  prod["txfecha_apertura"] = pd.to_datetime(prod["txfecha_apertura"], errors="coerce", dayfirst=True)



==================== PERFIL TRANSACCIONES ====================
Filas: 120,000 | Columnas: 14
Rango txfecha: 2024-01-01 00:00:00 -> 2024-06-30 00:00:00 | Nulos: 0
- txtotal: nulos=0 | p50=2000000.0 | p95=287474426.0 | max=291199371128
- txefectivo: nulos=0 | p50=0.0 | p95=6000000.0 | max=20722264376
- Top ofcodigo:
ofcodigo
544     8063
1114    7652
1115    4794
550     3972
1116    3879
- Top cargo:
cargo
CAJERO    117434
ASESOR      2566
- Top ctxcodigoproducto:
ctxcodigoproducto
76    61065
85    58935

==================== PERFIL PRODUCTOS ====================
Filas: 22,824 | Columnas: 10
Rango txfecha_apertura: 2023-01-01 00:00:00 -> 2024-06-26 00:00:00 | Nulos: 0
- monto: nulos=0 | p50=216026880.0 | p95=475017143.0 | max=17769126422
- plazo: nulos=0 | p50=270.0 | p95=540.0 | max=600
- Top ctxcodigoproducto:
ctxcodigoproducto
76    22824

==================== PERFIL CODIGOS_TRANSACCION ====================
Filas: 114 | Columnas: 3
- Top requiere_presencia_cliente:
requiere_presenc

In [4]:
# 12) SUBIR A POSTGRES
#   - STG: REPLACE
#   - FACT: APPEND SOLO NUEVAS
#   - AUD TX*: APPEND SOLO NUEVAS (filtradas anti-duplicado)
#   - APERTURA / ALERTAS USUARIO: REPLACE (práctico)
# ============================================================
subir_replace(trans, "stg_transacciones")
subir_replace(cod,   "stg_codigos_transaccion")
subir_replace(prod,  "stg_productos")
subir_replace(ofis,  "stg_oficinas")
subir_replace(cli,   "stg_clientes")
subir_replace(tipid, "stg_tipo_identificacion")

# FACT incremental
subir_append(fact_nuevo, "fact_transaccion")

# AUD TX DETALLE incremental (evita duplicar por tx_id+regla_id)
if not aud_anomalias_tx_detalle.empty:
    aud_det_nuevo = filtrar_existentes_por_llave(aud_anomalias_tx_detalle, "aud_anomalias_tx_detalle", ["tx_id", "regla_id"])
else:
    aud_det_nuevo = aud_anomalias_tx_detalle
subir_append(aud_det_nuevo, "aud_anomalias_tx_detalle")

# AUD TX consolidada incremental (evita duplicar por tx_id)
if not aud_anomalias_tx.empty:
    aud_tx_nuevo = filtrar_existentes_por_llave(aud_anomalias_tx, "aud_anomalias_tx", ["tx_id"])
else:
    aud_tx_nuevo = aud_anomalias_tx
subir_append(aud_tx_nuevo, "aud_anomalias_tx")

# Aperturas / Alertas usuario (REPLACE recomendado)
subir_replace(aud_anomalias_apertura, "aud_anomalias_apertura")
subir_replace(aud_alertas_usuario,    "aud_alertas_usuario")

print("\n🎉 LISTO: ETL incremental (FACT + AUD TX incremental) + tablas para Power BI")

✅ stg_transacciones (REPLACE): 120,000 filas
✅ stg_codigos_transaccion (REPLACE): 114 filas
✅ stg_productos (REPLACE): 22,824 filas
✅ stg_oficinas (REPLACE): 236 filas
✅ stg_clientes (REPLACE): 27,600 filas
✅ stg_tipo_identificacion (REPLACE): 6 filas
⚡ fact_transaccion (APPEND): 0 filas (sin nuevos)
⚡ aud_anomalias_tx_detalle (APPEND): 0 filas (sin nuevos)
⚡ aud_anomalias_tx (APPEND): 0 filas (sin nuevos)
✅ aud_anomalias_apertura (REPLACE): 15 filas
✅ aud_alertas_usuario (REPLACE): 501 filas

🎉 LISTO: ETL incremental (FACT + AUD TX incremental) + tablas para Power BI
